In [59]:
# download customer churn dataset from kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\devku\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1


## Task 1: Exploratory Data Analysis

### 1) Data Cleaning
#### 1.1) Handling empty values

In [60]:
import pandas as pd
import matplotlib as plt

data = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv") # load csv into dataframe
print("Dataset information:")
print(data.info()) # print info about the dataset

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null 

In [61]:
'''conduct basic check for null values in the dataset'''
print("Null-value check for each cell in the dataset:")
print(data.isnull())

Null-value check for each cell in the dataset:
      customerID  gender  SeniorCitizen  Partner  Dependents  tenure  \
0          False   False          False    False       False   False   
1          False   False          False    False       False   False   
2          False   False          False    False       False   False   
3          False   False          False    False       False   False   
4          False   False          False    False       False   False   
...          ...     ...            ...      ...         ...     ...   
7038       False   False          False    False       False   False   
7039       False   False          False    False       False   False   
7040       False   False          False    False       False   False   
7041       False   False          False    False       False   False   
7042       False   False          False    False       False   False   

      PhoneService  MultipleLines  InternetService  OnlineSecurity  ...  \
0            

In [62]:
'''conduct specific check for empty strings through regex while excluding numerical columns'''
print("Empty-string check for non-numeric columns:")
for i in data.select_dtypes(exclude='number').columns: # iterate through all numeric columns
    print(f'Column {i}: {data[i].str.contains(r"^\s*$").unique()}') # print value counts for each column

Empty-string check for non-numeric columns:
Column customerID: [False]
Column gender: [False]
Column Partner: [False]
Column Dependents: [False]
Column PhoneService: [False]
Column MultipleLines: [False]
Column InternetService: [False]
Column OnlineSecurity: [False]
Column OnlineBackup: [False]
Column DeviceProtection: [False]
Column TechSupport: [False]
Column StreamingTV: [False]
Column StreamingMovies: [False]
Column Contract: [False]
Column PaperlessBilling: [False]
Column PaymentMethod: [False]
Column TotalCharges: [False  True]
Column Churn: [False]


In [63]:
'''Identification of correlation between Tenure and TotalCharges'''
# check if str col has any alphabetical characters 
try:
    totalChargeAlphabets = data[data['TotalCharges'].str.contains(r'[a-zA-Z]|^\s*$')] # seperate rows with non-numeric and missing values in TotalCharges col
except AttributeError:
    print('"TotalCharges" is already a numeric column.')
except Exception as e:
    print(f'An error occurred: {e}')

print(totalChargeAlphabets['tenure'],totalChargeAlphabets['TotalCharges']) # print both tenure and totalCharges to view the connection


488     0
753     0
936     0
1082    0
1340    0
3331    0
3826    0
4380    0
5218    0
6670    0
6754    0
Name: tenure, dtype: int64 488      
753      
936      
1082     
1340     
3331     
3826     
4380     
5218     
6670     
6754     
Name: TotalCharges, dtype: str


In [64]:
'''replace TotalCharge from ' ' to 0 where tenure is 0'''
data.loc[(data['TotalCharges'] == ' ') & (data['tenure'] == 0), 'TotalCharges'] = f'{0.0}'
data['TotalCharges'] = data['TotalCharges'].astype(float)

print("TotalCharges data type after conversion:")
print(data['TotalCharges'].dtype) # check if TotalCharges is now a float
print("Rows with zero tenure after replacing empty TotalCharges values:")
print(data.loc[data['tenure'] == 0.0])

TotalCharges data type after conversion:
float64
Rows with zero tenure after replacing empty TotalCharges values:
      customerID  gender  SeniorCitizen Partner Dependents  tenure  \
488   4472-LVYGI  Female              0     Yes        Yes       0   
753   3115-CZMZD    Male              0      No        Yes       0   
936   5709-LVOEQ  Female              0     Yes        Yes       0   
1082  4367-NUYAO    Male              0     Yes        Yes       0   
1340  1371-DWPAZ  Female              0     Yes        Yes       0   
3331  7644-OMVMY    Male              0     Yes        Yes       0   
3826  3213-VVOLG    Male              0     Yes        Yes       0   
4380  2520-SGTTA  Female              0     Yes        Yes       0   
5218  2923-ARZLG    Male              0     Yes        Yes       0   
6670  4075-WKNIU  Female              0     Yes        Yes       0   
6754  2775-SEFEE    Male              0      No        Yes       0   

     PhoneService     MultipleLines InternetS

In [65]:
#'''Identification of non-numeric values in TotalCharges column'''
#
## check if str col has any alphabetical characters 
#try:
#    totalChargeAlphabets = data[data['TotalCharges'].str.contains(r'[a-zA-Z]|^\s*$')] # seperate rows with non-numeric and missing values in TotalCharges col
#    totalChargeNumeric = data[data['TotalCharges'].str.contains(r'[0-9]')] # seperate rows with numeric values in TotalCharges col
#except AttributeError:
#    print('"TotalCharges" is already a numeric column.')
#except Exception as e:
#    print(f'An error occurred: {e}')
#
#print("Rows with empty or non-numeric values in 'TotalCharges':")
#print(totalChargeAlphabets['TotalCharges']) # print info about the numeric TotalCharges col

In [66]:
#'''replacing non-numeric values in TotalCharges column with column mean after data type conversion to float'''
#
#TCMean = pd.to_numeric(data['TotalCharges'], errors='coerce').mean() # calculate mean of numeric TotalCharges col
#try:
#    mask = data['TotalCharges'].str.contains(r'[a-zA-Z]|^\s*$')
#    data.loc[mask, 'TotalCharges'] = f'{TCMean}'   # rows = mask, column = TotalCharges only
#except AttributeError:
#    print('"TotalCharges" is already a numeric column.')
#except Exception as e:
#    print(f'An error occurred: {e}')
#
#data['TotalCharges'] = data['TotalCharges'].astype(float)
#
#print(f"TotalCharges mean used to replace empty values: {TCMean}")
#print("Rows after replacing empty or non-numeric values in 'TotalCharges':")
#print(data.loc[mask, 'TotalCharges']) # print info about the numeric TotalCharges col

#### 1.2) Handle Duplicate Rows

In [ ]:
'''basic duplicate row check'''
print("Basic duplicate row check:")
print(data.duplicated().sum()) # check for duplicate rows if any are available

Basic duplicate row check:
0


In [68]:
'''check for duplicate rows if any are available'''
print("Checking for duplicate rows in the dataset...")
duplicates_before_id_removal = data.duplicated().sum()
print(f"---Duplicate rows before removing customerID: {duplicates_before_id_removal}")

# Remove customerID only if it exists
if 'customerID' in data.columns:
    data.drop(columns=['customerID'], inplace=True)
    print("The customerID column was removed because it identifies individual customers.")
else:
    print("The customerID column was not present in the dataset.")

duplicate_rows = data[data.duplicated()]
print(f"---Duplicate rows after removing customerID: {len(duplicate_rows)}")
if len(duplicate_rows) > 0:
    print("Duplicate rows found and will be removed.")
else:
    print("No duplicate rows were found.")

data.drop_duplicates(inplace=True) # remove duplicate rows if any are available

print(f"---Rows remaining after duplicate removal: {len(data)}")
print(f"---Duplicate rows remaining: {data.duplicated().sum()}")

Checking for duplicate rows in the dataset...
---Duplicate rows before removing customerID: 0
The customerID column was removed because it identifies individual customers.


---Duplicate rows after removing customerID: 22
Duplicate rows found and will be removed.
---Rows remaining after duplicate removal: 7021
---Duplicate rows remaining: 0


### 2) Data Visualization